# Lab 3 starter: the top-ten customer table

This notebook is standalone. The recap cell rebuilds `lines`, the joined table
Chapter 3 assembles in three separate steps, so you can start here without
re-running the chapter.

The lab prompt is at the end of Chapter 3, under **Exercises**, in the sections
headed *Build lab* and *Evaluate lab*.

**Build lab.** Build the top-ten customer table for 2025: filter `lines` to 2025,
group by `customer_id`, and compute total revenue, the number of distinct orders,
and the average discount. Join each customer's name, sort by revenue, and keep the
top ten. Then say which business types dominate the list.

**Evaluate lab.** Reconcile the table. Check that your full 2025 grouping accounts
for every 2025 dollar, and that joining the names added no rows. Then state what a
failure of each check would have meant.

## Running this notebook

Run the setup cell below first, then the recap cell, then work down. In Google
Colab nothing needs to be installed beyond that cell. Locally, use the
`pyba-core` environment from Appendix A.

You will enter your work in the cells marked `# TODO`. Everything else is
provided.

In [ ]:
# Setup. Run this cell once per session. It installs this lab's packages;
# on Google Colab it also fetches the course data.
%pip install -q pandas openpyxl
import sys
if "google.colab" in sys.modules:
    !git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Recap: the Chapter 3 table

The cell below reproduces `lines` as Chapter 3 leaves it: orders joined to product
details, a `margin_dollars` column, and the customer's region and business type
joined on. Run it before anything else.

In [ ]:
# --- Chapter 3 recap: run this cell first ---
import pandas as pd

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "pw_products.csv")
customers = pd.read_csv(DATA_DIR / "pw_customers.csv", parse_dates=["signup_date"])

product_cols = products[["sku", "product_name", "category", "unit_cost"]]
lines = orders.merge(product_cols, on="sku", how="left", validate="many_to_one")
lines["margin_dollars"] = lines["line_total"] - lines["quantity"] * lines["unit_cost"]
lines = lines.merge(customers[["customer_id", "region", "business_type"]],
                    on="customer_id", how="left", validate="many_to_one")

print(f"lines: {len(lines):,} rows, {lines['customer_id'].nunique()} customers")
lines.head(3)

## Step 1: filter to 2025

One line. `lines["order_date"]` is already a datetime column, so `.dt.year` gives
you the year to compare against. This step is provided, since the chapter covers
date filtering directly.

In [ ]:
lines_2025 = lines[lines["order_date"].dt.year == 2025]
print(f"{len(lines_2025):,} rows in 2025")

## Step 2: group by customer

Group `lines_2025` by `customer_id` and compute three columns in one `.agg` call:

| column | from |
|---|---|
| `revenue` | the sum of `line_total` |
| `n_orders` | the number of **distinct** `order_id` values |
| `avg_discount` | the mean of `discount_pct` |

Two hints. Use the named-aggregation form from the chapter,
`revenue=("line_total", "sum")`, which names each output column as you build it.
And `n_orders` is the one to think about: a customer's rows are order *lines*, so
counting rows would overcount. `("order_id", "nunique")` counts distinct orders.

Call `.reset_index()` at the end so that `customer_id` becomes a column again,
which the join in Step 3 needs.

In [ ]:
# TODO: group lines_2025 by customer_id and compute revenue, n_orders, avg_discount
by_customer = None

by_customer.head()

## Step 3: join the customer names

Merge `by_customer` with `customers` to bring in each customer's `name`.

Select **only** `customer_id` and `name` from `customers` before merging. `lines`
already carries `business_type`, and merging the whole customers frame would bring
in a duplicate copy of it. Pass `validate="many_to_one"` so that pandas raises an
error if the join would fan out rows, which is exactly the failure Step 5 checks
for.

In [ ]:
# TODO: merge by_customer with customers[["customer_id", "name"]]
named = None

named.head()

## Step 4: the top ten

Sort `named` by revenue, highest first, and keep ten rows. Then look at the
`business_type` column and answer the question the lab asks.

`business_type` is not in `by_customer`, since you grouped it away in Step 2. Pull
it back by merging from `customers`, or add it to the grouping keys in Step 2 (it
is constant within a customer, so grouping by both changes nothing).

In [ ]:
# TODO: sort by revenue, keep the top ten
top_ten = None

top_ten

**Which business types dominate the list?** Answer in one or two sentences here.

---

...

---

## Step 5 (Evaluate lab): reconcile the table

Two checks, both written as assertions.

**Check one: no dollars lost in the grouping.** The `revenue` column of
`by_customer`, summed over every customer, must equal the `line_total` of
`lines_2025`, summed over every row. Compare them rounded to the cent, as in
Chapter 2, because floating-point sums of money rarely match exactly.

**Check two: the join added no rows.** `by_customer` holds one row per customer.
After merging the names, `named` must still hold one row per customer, and the
same number of them. Compare `len()` before and after, and confirm that
`customer_id` has no duplicates.

In [ ]:
# TODO: check one, every 2025 dollar accounted for
assert ...

# TODO: check two, the join preserved one row per customer
assert ...
assert ...

print("Reconciled.")

## Step 6: say what a failure would have meant

For each of the two checks, write one sentence: if that assertion had failed, what
would have gone wrong?

The first check is about the grouping. The second is about the join, and the
chapter's discussion of `validate="many_to_one"` describes the failure it catches.

---

**If check one had failed:** ...

**If check two had failed:** ...

---

## Before you submit

Run **Restart and run all** and confirm the notebook executes top to bottom with no
errors. Then download as `.ipynb` and upload it to the Lab 3 dropbox.

If you used an AI assistant, add a one-line note naming the tool and what it
generated, per the course policy in Appendix D.